# 00 — Length-bin threshold analysis

**Purpose:** Determine and justify channel-level `length_bin` thresholds for all Core sources.

The resulting thresholds are written to `v2/config.py` and must be imported by every preparation notebook.

**Sources analysed:**

| Channel | Source | Location |
|---------|--------|----------|
| `sms` | SmishTank fraud | `gathered/smishtank_prepared.jsonl` |
| `sms` | Mendeley smishing fraud | `gathered/mendeley_smishing_prepared.jsonl` |
| `sms` | SMS ham | `raw/collected/sms_spam.jsonl` (label=ham) |
| `email` | Nazario phishing | `gathered/nazario_prepared.jsonl` |
| `email` | Nigerian 419 fraud | `gathered/nigerian_fraud_prepared.jsonl` |
| `email` | Enron ham | `raw/collected/enron.jsonl` (label=ham) |
| `email` | SpamAssassin ham | `raw/collected/spamassassin.jsonl` (label=ham) |
| `qa` | HC3 Finance human | `raw/collected/hc3_finance.jsonl` (role=human) |
| `qa` | HC3 Finance chatgpt | `raw/collected/hc3_finance.jsonl` (role=chatgpt) |

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd

BASE     = Path('/Users/askar/projects/antifraud-deepfake-detection/v2')
GATHERED = BASE / 'data/interim/gathered'
RAW_COL  = BASE / 'data/raw/collected'

TOKEN_RE = re.compile(r'\w+|[^\w\s]', re.UNICODE)


def tok(text: str) -> int:
    return len(TOKEN_RE.findall(text or ''))


def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(l) for l in path.read_text(encoding='utf-8').splitlines() if l.strip()]


def percentile_row(name: str, channel: str, token_lengths: list[int]) -> dict:
    s = sorted(token_lengths)
    n = len(s)
    def p(q): return s[int(n * q)] if n else 0
    return {
        'source': name,
        'channel': channel,
        'n': n,
        'min': s[0] if s else 0,
        'p10': p(0.10), 'p25': p(0.25), 'p50': p(0.50),
        'p75': p(0.75), 'p90': p(0.90), 'p95': p(0.95),
        'max': s[-1] if s else 0,
    }


rows_meta: list[dict] = []      # percentile table rows
raw_tokens: dict[str, list[int]] = {}  # source_name → token lengths

# ── SMS ──────────────────────────────────────────────────────────────────────
for fname, label in [
    ('smishtank_prepared.jsonl',          'SmishTank fraud (sms)'),
    ('mendeley_smishing_prepared.jsonl',  'Mendeley fraud (sms)'),
]:
    data = load_jsonl(GATHERED / fname)
    tl = [r.get('token_length', tok(r['text'])) for r in data]
    raw_tokens[label] = tl
    rows_meta.append(percentile_row(label, 'sms', tl))

sms_raw = load_jsonl(RAW_COL / 'sms_spam.jsonl')
sms_ham_tl = [tok(r['text']) for r in sms_raw if r.get('label') == 'ham']
raw_tokens['SMS ham (sms)'] = sms_ham_tl
rows_meta.append(percentile_row('SMS ham (sms)', 'sms', sms_ham_tl))

# ── Email ─────────────────────────────────────────────────────────────────────
for fname, label in [
    ('nazario_prepared.jsonl',        'Nazario phishing (email)'),
    ('nigerian_fraud_prepared.jsonl', 'Nigerian fraud (email)'),
]:
    data = load_jsonl(GATHERED / fname)
    tl = [r.get('token_length', tok(r['text'])) for r in data]
    raw_tokens[label] = tl
    rows_meta.append(percentile_row(label, 'email', tl))

enron_raw = load_jsonl(RAW_COL / 'enron.jsonl')
enron_ham_tl = [tok(r['text']) for r in enron_raw if r.get('label') == 'ham']
raw_tokens['Enron ham (email)'] = enron_ham_tl
rows_meta.append(percentile_row('Enron ham (email)', 'email', enron_ham_tl))

sa_raw = load_jsonl(RAW_COL / 'spamassassin.jsonl')
sa_ham_tl = [tok(r['text']) for r in sa_raw if r.get('label') == 'ham']
raw_tokens['SpamAssassin ham (email)'] = sa_ham_tl
rows_meta.append(percentile_row('SpamAssassin ham (email)', 'email', sa_ham_tl))

# ── QA ───────────────────────────────────────────────────────────────────────
hc3_raw = load_jsonl(RAW_COL / 'hc3_finance.jsonl')
hc3_human_tl   = [tok(r['text']) for r in hc3_raw if r.get('role') == 'human']
hc3_chatgpt_tl = [tok(r['text']) for r in hc3_raw if r.get('role') == 'chatgpt']
raw_tokens['HC3 human (qa)']   = hc3_human_tl
raw_tokens['HC3 chatgpt (qa)'] = hc3_chatgpt_tl
rows_meta.append(percentile_row('HC3 human (qa)',   'qa', hc3_human_tl))
rows_meta.append(percentile_row('HC3 chatgpt (qa)', 'qa', hc3_chatgpt_tl))

df_meta = pd.DataFrame(rows_meta)
print('Sources loaded:')
print(df_meta[['source', 'channel', 'n']].to_string(index=False))

In [ ]:
# ── Percentile table per channel ──────────────────────────────────────────────
cols = ['source', 'n', 'min', 'p10', 'p25', 'p50', 'p75', 'p90', 'p95', 'max']

for ch in ['sms', 'email', 'qa']:
    sub = df_meta[df_meta['channel'] == ch][cols].reset_index(drop=True)
    print(f'\n── Channel: {ch} ────────────────────────────────────────────────')
    print(sub.to_string(index=False))

# Nazario outlier diagnostic
naz_tl = raw_tokens['Nazario phishing (email)']
extreme = sorted([t for t in naz_tl if t > 5000], reverse=True)
print(f'\nNazario outliers > 5000 tokens: {len(extreme)}')
if extreme:
    print('  token counts:', extreme[:10])

In [ ]:
# ── Proposed thresholds and resulting bin distributions ───────────────────────
#
# Design rationale
# ----------------
# Channel-level (not universal, not per-source) because:
#   1. SMS and email are incomparable in token scale.
#   2. Channel is already the primary semantic axis in dataset_design_final.md.
#   3. A single shared threshold would collapse all SMS to 'short' on the
#      email scale, destroying within-SMS stratification.
#   4. Per-source thresholds break cross-source comparison within a channel
#      and caused the smishtank / mendeley inconsistency (April 2026 audit).
#
# Threshold selection
# -------------------
# SMS  : fraud SMS p50=32, p75=45; ham SMS p75=24, p90=37.
#        short <20  → covers most ham (p50=14) and very short fraud;
#        medium 20–59 → bulk of fraud SMS;
#        long  ≥60  → longer fraud messages (max ~200).
#
# Email: wide range 30–300k (outlier capped at 5000 for real distribution).
#        short <100  → p10–p25 of fraud/ham email;
#        medium 100–399 → modal range across all email sources;
#        long  ≥400  → Enron p90=660, Nazario p90=471.
#
# QA   : HC3 human p25=87, p50=156, p75=274; chatgpt narrower (p25=192).
#        short <75  → ~25% of human answers;
#        medium 75–249 → modal human range;
#        long  ≥250 → top ~25% of human + most chatgpt.

THRESHOLDS = {
    'sms':   {'short': 20,  'medium': 60},
    'email': {'short': 100, 'medium': 400},
    'qa':    {'short': 75,  'medium': 250},
}


def apply_bins(token_lengths: list[int], channel: str) -> dict[str, int]:
    t = THRESHOLDS[channel]
    counts: dict[str, int] = {'short': 0, 'medium': 0, 'long': 0}
    for tl in token_lengths:
        if tl < t['short']:   counts['short']  += 1
        elif tl < t['medium']: counts['medium'] += 1
        else:                  counts['long']   += 1
    return counts


bin_rows = []
for name, tl in raw_tokens.items():
    ch = 'sms' if '(sms)' in name else ('email' if '(email)' in name else 'qa')
    # exclude extreme outliers from email counts (cap at 5000 for display)
    tl_capped = [min(t, 5000) for t in tl] if ch == 'email' else tl
    counts = apply_bins(tl_capped, ch)
    n = len(tl)
    bin_rows.append({
        'source': name, 'channel': ch, 'n': n,
        'short': counts['short'],
        'medium': counts['medium'],
        'long': counts['long'],
        'short%': f"{counts['short']/n*100:.0f}%" if n else '-',
        'medium%': f"{counts['medium']/n*100:.0f}%" if n else '-',
        'long%': f"{counts['long']/n*100:.0f}%" if n else '-',
    })

df_bins = pd.DataFrame(bin_rows)
print('Proposed thresholds:')
for ch, t in THRESHOLDS.items():
    print(f'  {ch:5s}: short < {t["short"]:3d}  |  medium {t["short"]}–{t["medium"]-1:3d}  |  long >= {t["medium"]}')

print()
for ch in ['sms', 'email', 'qa']:
    sub = df_bins[df_bins['channel'] == ch][['source', 'n', 'short', 'medium', 'long', 'short%', 'medium%', 'long%']]
    print(f'\n── {ch} bin distributions ──────────────────────────────────────────')
    print(sub.to_string(index=False))

In [ ]:
# ── Write v2/config.py ────────────────────────────────────────────────────────
# config.py is the single source of truth for length_bin thresholds.
# All preparation notebooks import length_bin() from it.
import sys
sys.path.insert(0, str(BASE))
import importlib, config as cfg
importlib.reload(cfg)

# Verify the already-written config.py matches the thresholds derived above.
assert cfg.LENGTH_BIN_THRESHOLDS == THRESHOLDS, (
    f'config.py thresholds differ from analysis!\n'
    f'  config.py : {cfg.LENGTH_BIN_THRESHOLDS}\n'
    f'  analysis  : {THRESHOLDS}'
)

# Smoke-test the length_bin() function
assert cfg.length_bin(5,   'sms')   == 'short'
assert cfg.length_bin(30,  'sms')   == 'medium'
assert cfg.length_bin(100, 'sms')   == 'long'
assert cfg.length_bin(50,  'email') == 'short'
assert cfg.length_bin(200, 'email') == 'medium'
assert cfg.length_bin(500, 'email') == 'long'
assert cfg.length_bin(30,  'qa')    == 'short'
assert cfg.length_bin(150, 'qa')    == 'medium'
assert cfg.length_bin(300, 'qa')    == 'long'

print('config.py verified ✓')
print(f'Path: {BASE / "config.py"}')
print()
print('Thresholds in effect:')
for ch, t in cfg.LENGTH_BIN_THRESHOLDS.items():
    print(f'  {ch:5s}: short < {t["short"]:3d}  |  medium {t["short"]}–{t["medium"]-1:3d}  |  long >= {t["medium"]}')

print()
print('All preparation notebooks should import via:')
print("  import sys; sys.path.insert(0, str(BASE))")
print("  from config import length_bin as compute_length_bin")